# Clasificación de intenciones bancarias (versión simple)

Hola. Este es tu **primer proyecto de IA** y vas a resolver un problema real: dado un mensaje de un cliente del banco, adivinar su **intención** (por ejemplo: `card_arrival`, `lost_or_stolen_card`).

Hay **77 intenciones** distintas. Usaremos solo `sklearn` + `pandas` + `matplotlib/seaborn`, todo en CPU en 1-3 minutos. Sin descargas de modelos.

**Plan:** 1) cargar datos · 2) explorar con gráficos · 3) modelo base (TF-IDF + Regresión Logística) · 4) segundo modelo (TF-IDF + MLP) · 5) cierre: por qué existen los transformers.


In [ ]:
import pandas as pd  # tablas de datos
import matplotlib.pyplot as plt  # gráficos
import seaborn as sns  # gráficos más lindos
from sklearn.pipeline import make_pipeline  # une pasos en cadena
from sklearn.feature_extraction.text import TfidfVectorizer  # texto -> números
from sklearn.linear_model import LogisticRegression  # modelo base
from sklearn.neural_network import MLPClassifier  # segundo modelo
from sklearn.metrics import accuracy_score, f1_score, classification_report  # métricas


## 1. Cargar los datos

- **Leer los CSV:** cargamos `train` (para aprender) y `test` (para evaluar al final, con datos nuevos).
- **Ver el tamaño (`shape`):** confirma cuántas filas y columnas tenemos.
- **Contar intenciones:** verifica que hay 77 categorías distintas.


In [ ]:
train_df = pd.read_csv("datasets/banking77_train.csv")  # datos de entrenamiento
test_df = pd.read_csv("datasets/banking77_test.csv")  # datos de prueba
print(train_df.shape, test_df.shape)  # filas y columnas
print(train_df.head(3).to_string())  # mira las primeras filas


In [ ]:
n_intents = train_df["category"].nunique()  # cuenta intenciones únicas
print("Intents:", n_intents)  # debe ser 77
print(sorted(train_df["category"].unique())[:5])  # mira 5 nombres


## 2. Explorar los datos (EDA)

Antes de entrenar, miramos los datos. Un buen modelo empieza por entender el problema.


In [ ]:
counts = train_df["category"].value_counts().head(10)  # top-10 intenciones
plt.figure(figsize=(8, 4))  # tamaño del gráfico
sns.barplot(x=counts.values, y=counts.index)  # barras horizontales
plt.xlabel("examples")  # eje x
plt.title("Top-10 intents")  # título
plt.tight_layout(); plt.show()  # muestra el gráfico


**Qué veo:** las 10 intenciones más frecuentes tienen entre ~130 y ~190 ejemplos cada una.
**Por qué importa:** si todo está parecido, el modelo no puede "hacer trampa" prediciendo siempre la misma clase; el dataset está bastante balanceado.


In [ ]:
lengths = train_df["text"].str.len()  # largo de cada consulta
plt.figure(figsize=(8, 4))  # tamaño del gráfico
plt.hist(lengths, bins=30)  # histograma de largos
plt.xlabel("characters per query")  # eje x
plt.ylabel("number of queries")  # eje y
plt.title("Query length")  # título
plt.tight_layout(); plt.show()  # muestra el gráfico


**Qué veo:** la mayoría de los mensajes son cortos (entre ~20 y ~150 caracteres).
**Por qué importa:** con textos cortos, contar palabras (TF-IDF) ya es una señal fuerte y barata.


In [ ]:
for intent in ["card_arrival", "lost_or_stolen_card", "exchange_rate"]:  # elige 3 intenciones
    sample = train_df[train_df["category"] == intent].iloc[0]["text"]  # primer ejemplo
    print(f"[{intent}] {sample}")  # muestra el ejemplo


## 3. Modelo base: TF-IDF + Regresión Logística

Convertimos cada texto en conteos de palabras ponderados (TF-IDF) y entrenamos una Regresión Logística.

**Fuga de datos (*leakage*):** ajustamos (`fit`) el TF-IDF SOLO con `train`. Si usáramos `test`, el modelo "espiaría" las respuestas del examen y la nota sería mentirosa.


In [ ]:
baseline = make_pipeline(  # une vectorizador + modelo
    TfidfVectorizer(max_features=8000),  # texto -> 8000 columnas
    LogisticRegression(max_iter=1000),  # clasificador lineal
)  # fin del pipeline
baseline.fit(train_df["text"], train_df["category"])  # aprende SOLO de train
print("trained")  # confirmación


In [ ]:
pred_base = baseline.predict(test_df["text"])  # predice datos nuevos
acc_base = accuracy_score(test_df["category"], pred_base)  # aciertos totales
f1_base = f1_score(test_df["category"], pred_base, average="macro")  # promedio por intención
print(f"Baseline accuracy: {acc_base:.4f} | macro-F1: {f1_base:.4f}")  # muestra métricas


In [ ]:
report = classification_report(test_df["category"], pred_base, output_dict=True, zero_division=0)  # F1 por intención
scores = sorted(((k, v["f1-score"]) for k, v in report.items() if k not in ("accuracy", "macro avg", "weighted avg")), key=lambda x: x[1])  # ordena por F1
print("Worst 3:", [(k, round(v, 3)) for k, v in scores[:3]])  # 3 peores
print("Best 3:", [(k, round(v, 3)) for k, v in scores[-3:]])  # 3 mejores


## 4. Segundo modelo: TF-IDF + MLP (1 capa oculta de 100)

Mismo texto numérico (TF-IDF), pero el clasificador ahora es una red pequeña: una capa oculta de 100 neuronas que puede combinar palabras de forma no lineal.


In [ ]:
mlp = make_pipeline(  # mismo vectorizador, otro clasificador
    TfidfVectorizer(max_features=8000),  # texto -> 8000 columnas
    MLPClassifier(hidden_layer_sizes=(100,), max_iter=20, random_state=42),  # red pequeña
)  # fin del pipeline
mlp.fit(train_df["text"], train_df["category"])  # aprende SOLO de train
print("trained")  # confirmación


In [ ]:
pred_mlp = mlp.predict(test_df["text"])  # predice datos nuevos
acc_mlp = accuracy_score(test_df["category"], pred_mlp)  # aciertos totales
f1_mlp = f1_score(test_df["category"], pred_mlp, average="macro")  # promedio por intención
print(f"MLP accuracy: {acc_mlp:.4f} | macro-F1: {f1_mlp:.4f}")  # muestra métricas
comparison = pd.DataFrame({"model": ["TF-IDF + LogReg", "TF-IDF + MLP(100)"], "accuracy": [acc_base, acc_mlp], "macro-F1": [f1_base, f1_mlp]})  # tabla comparativa
print(comparison.to_string(index=False))  # muestra la tabla


**Qué cambió:** solo el clasificador (lineal → red de 1 capa); la representación del texto es la misma.
**Por qué difiere:** la capa oculta combina señales de varias palabras a la vez, así que separa intenciones parecidas un poco mejor.
**Resultado:** el MLP supera al base por ~2 puntos con el mismo TF-IDF y sigue corriendo en CPU en segundos.


## 5. Cierre: ¿para qué existe un transformer + fine-tuning?

TF-IDF solo **cuenta palabras sueltas**: no entiende orden ni contexto ("no llegó mi tarjeta" vs "llegó mi tarjeta" comparten palabras pero dicen lo opuesto). Un transformer (BERT y familia) **lee la frase completa** y entiende el contexto de cada palabra.

**¿Cuándo alcanza el base?** Si tus mensajes son cortos y directos, y ya mides ~0.88–0.90 de accuracy en CPU en segundos, el base es suficiente y gratis.

**¿Cuándo pagar la GPU?** Cuando hay muchas intenciones parecidas, ironía, o frases largas donde el orden cambia el sentido; ahí el contexto del transformer justifica su costo (GPU, minutos/horas, más código).

**Conexión con el proyecto principal (sin re-ejecutar nada):** en `banking_classification_BERT.ipynb`, con su configuración completa, el MLP clásico llegó a **0.845** y el transformer con LoRA a **0.780**. Moraleja honesta: en este dataset el base clásico ya es muy fuerte, y el transformer solo vale la pena si necesitas ese entendimiento extra de contexto y puedes pagar su costo.

**Tu próximo paso:** si quieres seguir, abre el notebook principal y compara su pipeline (tokenizador + fine-tuning) con el TF-IDF de hoy. Ya sabes leer sus métricas.
